# Polygon REST API Demo - 9/15/26

### What does this notebook cover?

**Stocks**
- Pulling stock prices- both historical and live- on different timeframes
- Pulling corporate actions / company metadata
- Full market snapshots
- Technical indicators
- Earnings statements
- Current financial ratios

**Options**
- Pulling contracts- both open and closed- with different expiration date ranges
- Pulling price history and greeks on contracts 
- Building a simple options chain dataframe

**News, Sentiment, and other**
- News headlines 
- Comprehensive list of news articles
- Binned sentiment (`positive | neutral | negative`)
- Live and projected market status

**Live examples:** AAPL for company research - SPY for market and options analysis  
**Goal:** learn the useful endpoints, understand our actual entitlements, and leave with reusable Python for your projects.

## Environment Setup 

### Massive API Key 

Look in the Discord for the API key if you do not have it.

### Environment file (`.env`) setup

Create a new environment file in the same directory as this notebook:

Windows, MacOS, and Linux:
```cmd
echo POLYGON_API_KEY=[key here!] > .env
```

### External Libraries 

We use a variety of common external Python libraries to handle data, make requests, and display interactive charts.

| Library | Use |
|---|---|
| `requests` | Make HTTP requests| 
| `requests-cache` | dynamically cache certain HTTP requests | 
| `pandas` | tabular data handling | 
| `numpy` | vector math |
| `python-dotenv` | load environment variables from `.env` |
| `plotly` | render dynamic, interactive graphs and charts |

If you do not have these libraries installed, install them by uncommenting the line starting with '#' and running the cell.

In [ ]:
# environment setup 
%pip install requests requests-cache pandas numpy python-dotenv plotly "nbformat>=4.2.0"

import os
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
import requests_cache
from dotenv import load_dotenv
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [ ]:
BASE_URL = os.getenv("POLYGON_BASE_URL", "https://api.polygon.io").rstrip("/")
CACHE_MODE = os.getenv("POLYGON_CACHE_MODE", "live").lower()  # auto | live | cache
CACHE_DIR = Path(os.getenv("POLYGON_CACHE_DIR", ".polygon_demo_cache"))
REQUEST_TIMEOUT = 25

# replace with your .env file path
ENV = Path(
    ".env"
)
if ENV.exists():
    load_dotenv(ENV, override=False)
load_dotenv(override=False)

API_KEY = os.getenv("POLYGON_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "POLYGON_API_KEY is missing. Export it in your shell or put it in a local .env file."
    )

# caching setup: persistent sqlite cache only accepting GET requests with successful return 
CACHE_DIR.mkdir(parents=True, exist_ok=True)
session = requests_cache.CachedSession(
    cache_name=str(CACHE_DIR / "polygon_http_cache"),
    backend="sqlite",
    expire_after=timedelta(seconds=300),
    allowable_methods=("GET",),
    allowable_codes=(200,),
)

# important: keep plain text API KEY out of any cached URLs
# alternatively, use `ignored_parameters=["apiKey"]` when creating the cached session.
session.headers.update({"Authorization": f"Bearer {API_KEY}", "User-Agent": "gtsf-polygon-demo/1.0"})

print(f"Configured {BASE_URL} - cache mode={CACHE_MODE}")
print(Path.cwd())
print(ENV.resolve())

### API Call Wrapper 

Why use an API call wrapper?
The api_get() function provides one consistent way to make every Polygon API request in the notebook.
Without a wrapper, every endpoint would need to repeat the same setup:
- Construct the complete URL
- Attach the API key securely
- Remove unused parameters
- Configure the request timeout
- Apply the appropriate cache duration
- Check the HTTP response
- Extract useful API error messages
- Identify whether the response came from Polygon or the local cache
- Convert the response to JSON

---

Centralizing this logic makes the endpoint-specific functions much simpler. For example:
```python3 
payload = api_get(
    "/v3/reference/tickers/AAPL"
)
```

or 

```python3 
payload = api_get(
    "/v3/reference/options/contracts",
    {
        "underlying_ticker": "SPY",
        "expired": "false",
        "limit": 1000,
    },
)
```

...is a lot cleaner than messy key handling, param cleanup, etc. at every external call point. 

---

Benefits
- secure auth: API key is uniformly attached in the header, which isn't cached in this example. 
- consistent URL construction: one spot to change if/when the massive endpoint changes 
- clean params: only provided params are cached
- consistent caching: `_cache_ttl` can dynamically cache for each type of request
- clear error messages
- cleaner code
- maintenence 
- ...

In [ ]:
class PolygonAPIError(RuntimeError):
    '''An API error that intentionally omits credentials and full request URLs.'''


def _cache_ttl(path: str) -> int:
    '''
    Choose freshness by data type.
    
    Used to have different data refresh on different time periods.
    '''
    if "snapshot" in path or "marketstatus/now" in path:
        return 30
    if any(part in path for part in ("financials", "reference", "dividends", "splits", "holidays")):
        return 43_200 # 12 hours
    return 300


def api_get(
    path: str,
    params: dict[str, Any] | None = None,
    cache_mode: str = CACHE_MODE,
    *,
    quiet: bool = False,
) -> dict | list:
    '''GET one Polygon endpoint with header auth and live/cache controls.

    cache_mode = "auto" | "live | "cache" 
    '''
    mode = cache_mode.lower()
    if mode not in {"auto", "live", "cache"}:
        raise ValueError("cache_mode must be 'auto', 'live', or 'cache'")
    clean_params = {k: v for k, v in (params or {}).items() if v is not None}

    # try to provide shortened urls, eg. path="/v2/snapshot/locale/us/markets/stocks/tickers"
    url = path if path.startswith("http") else f"{BASE_URL}/{path.lstrip('/')}"
    kwargs = {
        "params": clean_params,
        "timeout": REQUEST_TIMEOUT,
        "expire_after": _cache_ttl(path),
        "force_refresh": mode == "live",
        "only_if_cached": mode == "cache",
    }

    response = session.get(url, **kwargs)

    if response.status_code == 504 and mode == "cache":
        raise PolygonAPIError(f"No cached response exists for {path}")

    elif response.status_code != 200:
        detail = response.reason

        if "application/json" in response.headers.get("Content-Type", ""):
            error_body = response.json()
            detail = error_body.get("error") or error_body.get("message") or detail
        raise PolygonAPIError(f"{path} returned HTTP {response.status_code}: {detail}")

    source = "CACHE" if getattr(response, "from_cache", False) else "LIVE"
    if not quiet:
        print(f"[{source}] {path}")
    return response.json()


def paginate(
    path: str,
    params: dict[str, Any] | None = None,
    max_pages: int | None = None,
    cache_mode: str = CACHE_MODE,
) -> list[dict]:
    '''Follow Polygon's next_url pagination and return one list of records.'''
    records, page, next_path, next_params = [], 0, path, params
    while next_path and (max_pages is None or page < max_pages):
        payload = api_get(next_path, next_params, cache_mode, quiet=page > 0)
        page_results = payload.get("results", []) if isinstance(payload, dict) else payload
        records.extend(page_results if isinstance(page_results, list) else [page_results])
        next_path = payload.get("next_url") if isinstance(payload, dict) else None
        next_params = None
        page += 1
    return records


def results_frame(payload: dict | list, record_path: str | None = None) -> pd.DataFrame:
    '''Normalize common Polygon response shapes into a DataFrame.'''
    if isinstance(payload, list):
        records = payload
    elif record_path:
        records = payload.get(record_path, [])
    else:
        records = payload.get("results", payload.get("ticker", payload))
    if isinstance(records, dict):
        records = [records]
    return pd.json_normalize(records)


def unix_to_eastern(values: Iterable, unit: str = "ms") -> pd.Series:
    parsed = pd.to_datetime(values, unit=unit, utc=True, errors="coerce")
    return parsed.dt.tz_convert("America/New_York") if isinstance(parsed, pd.Series) else parsed.tz_convert("America/New_York")

## Why use stock, indicator, and financial-data helpers?

These helper functions sit above `api_get()` and translate research questions into specific Massive API requests.

`api_get()` handles the shared request mechanics, while these functions handle endpoint-specific concerns such as:

- Constructing the correct endpoint path
- Formatting filters and parameters
- Converting API results into pandas DataFrames
- Renaming abbreviated fields
- Converting timestamps
- Validating the requested data type
- Sorting observations into a useful order

This separation keeps the analysis cells concise and makes the functions easy to reuse.

### Standardizing aggregate-bar columns

Polygon uses abbreviated field names in aggregate-bar responses:

- `t`: timestamp
- `o`: open
- `h`: high
- `l`: low
- `c`: close
- `v`: volume
- `vw`: volume-weighted average price
- `n`: number of transactions
- `T`: ticker

The `BAR_COLUMNS` mapping converts these API-specific names into readable DataFrame columns:

```python
BAR_COLUMNS = {
    "t": "timestamp",
    "o": "open",
    "h": "high",
    "l": "low",
    "c": "close",
    "v": "volume",
    "vw": "vwap",
    "n": "transactions",
    "T": "ticker",
}
```

### Historical stock bars
`get_stock_bars()` provides one interface to get OHLCV data at different frequencies. 

It: 
- converts tickers to uppercase
- builds the endpoint path
- sends the request via `api_get()`
- returns results chronologicaly
- normalizes column names to `BAR_COLUMNS`

The same function can be used to retrieve daily, weekly, hourly, or intraday data by changing `multiplier` and `timespan`.

```python 
get_stock_bars(
    "AAPL",
    start="2025-01-01",
    end="2025-12-31",
    multiplier=1,
    timespan="day",
)
```
returns daily bars for a year, while: 
```python 
get_stock_bars(
    "AAPL",
    start="2026-09-14",
    end="2026-09-14",
    multiplier=5,
    timespan="minute",
)
```
returns 5 minute bars on 9/14/2026.

### Current stock snapshots
`get_stock_snapshot()` retrieves a consolidated view of the latest available information for one stock.

The response contains:
- Current-day aggregate
- Latest minute aggregate
- Previous-day aggregate
- Fair Market Value
- Daily price change
- Latest trade
- Latest quote

### Technical indicators
`get_indicator()` provides one interface for four Polygon indicator endpoints:
- Simple moving average
- Exponential moving average
- MACD
- Relative Strength Index

The function validates kind, constructs the appropriate endpoint, and applies shared filters such as:
- Ticker
- Timespan
- Window
- Start and end dates
- Input price series
- Split adjustment
- Result limit

This lets the analysis use a consistent pattern:
```python 
sma20 = get_indicator("sma", "AAPL", window=20)
rsi14 = get_indicator("rsi", "AAPL", window=14)
```
instead of implementing a separate request function for every indicator.


### Financial statements
`get_financial_statements()` provides a common interface for:
- Income statements
- Balance sheets
- Cash-flow statements

The function validates the requested statement, retrieves quarterly or annual records, converts them into a DataFrame, and sorts them chronologically by `period_end`.

This makes it easy to switch between financial statements:
```python 
income = get_financial_statements(
    "AAPL",
    statement="income",
    timeframe="quarterly",
)

balance = get_financial_statements(
    "AAPL",
    statement="balance",
    timeframe="annual",
)
```
When conducting historical analysis, use `filing_date` to determine when each statement became publicly available and avoid look-ahead bias.

### Current financial ratios
`get_ratios()` retrieves the latest available financial-ratio record for a company and returns it.

The response can include:
- Valuation multiples
- Profitability ratios
- Liquidity ratios
- Leverage ratios
- Free cash flow
- Enterprise value
- Market capitalization
- Dividend yield

For example:
```python 
aapl_ratios = get_ratios("AAPL")
```
The ratios endpoint is a current snapshot. It does not provide a historical ratio time series.

In [ ]:
BAR_COLUMNS = {"t": "timestamp", "o": "open", "h": "high", "l": "low", "c": "close",
               "v": "volume", "vw": "vwap", "n": "transactions", "T": "ticker"}

def get_stock_bars(ticker, start, end, multiplier=1, timespan="day", adjusted=True, limit=50_000):
    path = f"/v2/aggs/ticker/{ticker.upper()}/range/{multiplier}/{timespan}/{start}/{end}"
    frame = results_frame(api_get(path, {"adjusted": str(adjusted).lower(), "sort": "asc", "limit": limit}))
    frame = frame.rename(columns=BAR_COLUMNS)
    if "timestamp" in frame:
        frame["timestamp"] = unix_to_eastern(frame["timestamp"])
    return frame


def get_stock_snapshot(ticker):
    """
    Get the current minute, day, and previous day's aggregate, as well as the last trade 
    and quote for a single traded stock ticker.

    see: https://massive.com/docs/rest/stocks/snapshots/single-ticker-snapshot
    """

    payload = api_get(f"/v2/snapshot/locale/us/markets/stocks/tickers/{ticker.upper()}")
    return payload.get("ticker", {})

def get_indicator(kind, ticker, timespan="day", window=None, start=None, end=None,
                  series_type="close", adjusted=True, limit=5_000):
    kind = kind.lower()
    if kind not in {"sma", "ema", "macd", "rsi"}:
        raise ValueError("kind must be sma, ema, macd, or rsi")
    params = {"timespan": timespan, "window": window, "timestamp.gte": start,
              "timestamp.lte": end, "series_type": series_type,
              "adjusted": str(adjusted).lower(), "order": "asc", "limit": limit}
    payload = api_get(f"/v1/indicators/{kind}/{ticker}", params)
    values = payload.get("results", {}).get("values", [])
    frame = pd.DataFrame(values)
    if "timestamp" in frame:
        frame["timestamp"] = unix_to_eastern(frame["timestamp"])
    return frame

def get_financial_statements(ticker, statement="income", timeframe="quarterly", limit=8):
    endpoints = {"income": "income-statements", "balance": "balance-sheets",
                 "cash_flow": "cash-flow-statements"}
    if statement not in endpoints:
        raise ValueError(f"statement must be one of {list(endpoints)}")
    rows = paginate(f"/stocks/financials/v1/{endpoints[statement]}",
                    {"tickers": ticker.upper(), "timeframe": timeframe,
                     "sort": "period_end.desc", "limit": limit}, max_pages=1)
    frame = pd.DataFrame(rows)
    if "period_end" in frame:
        frame = frame.sort_values("period_end")
    return frame

def get_ratios(ticker):
    return results_frame(api_get("/stocks/financials/v1/ratios", {"ticker": ticker.upper(), "limit": 1}))

## Why use options and sentiment helpers?
For the sake of brevity, long descriptions of these functions aren't provided. Most of them are just calling using `api_get()` and parsing the response. 

These functions convert Polygon’s nested options, news, and short-interest responses into analysis-ready DataFrames.

- `list_option_contracts()` finds contracts by underlying, expiration, strike, type, and expiration status.
- `_flatten_option()` converts each nested option snapshot into one flat row containing price data, IV, Greeks, open interest, and contract details.
- `get_option_chain()` retrieves the chain and calculates useful fields such as DTE, moneyness, intrinsic value, extrinsic value, volume/open-interest ratio, and delta-adjusted exposure.
- `get_option_bars()` retrieves historical OHLCV data for one option contract.
- `get_news()` retrieves recent ticker-specific articles and sentiment information.
- `get_short_interest()` retrieves and chronologically sorts reported short-interest history.

Together, these helpers let later cells analyze options and sentiment without repeatedly parsing nested JSON or rebuilding endpoint parameters.

In [ ]:

# finds all option contracts with expiration_gte <= expiration <= expiration_lte AND strike_gte <= strike <= strike_lte
def list_option_contracts(underlying, expiration_gte=None, expiration_lte=None,
                          strike_gte=None, strike_lte=None, contract_type=None,
                          expired=False, limit=1_000):
    params = {"underlying_ticker": underlying.upper(), "expiration_date.gte": expiration_gte,
              "expiration_date.lte": expiration_lte, "strike_price.gte": strike_gte,
              "strike_price.lte": strike_lte, "contract_type": contract_type,
              "expired": str(expired).lower(), "sort": "expiration_date", "order": "asc",
              "limit": min(limit, 1_000)}

    # keep getting pages until length of results is >= limit or no more results exist
    return pd.DataFrame(paginate("/v3/reference/options/contracts", params,
                                 max_pages=max(1, int(np.ceil(limit / 1_000)))))


def _flatten_option(row):
    details, day_data = row.get("details", {}), row.get("day", {})
    greeks, underlying = row.get("greeks", {}), row.get("underlying_asset", {})
    quote, trade = row.get("last_quote", {}), row.get("last_trade", {})
    return {
        "ticker": details.get("ticker"), "contract_type": details.get("contract_type"),
        "expiration": details.get("expiration_date"), "strike": details.get("strike_price"),
        "shares_per_contract": details.get("shares_per_contract", 100),
        "open": day_data.get("open"), "high": day_data.get("high"), "low": day_data.get("low"),
        "close": day_data.get("close"), "volume": day_data.get("volume"), "vwap": day_data.get("vwap"),
        "fmv": row.get("fmv"), "break_even": row.get("break_even_price"),
        "iv": row.get("implied_volatility"), "open_interest": row.get("open_interest"),
        "delta": greeks.get("delta"), "gamma": greeks.get("gamma"),
        "theta": greeks.get("theta"), "vega": greeks.get("vega"),
        "underlying_price": underlying.get("price"), "bid": quote.get("bid"),
        "ask": quote.get("ask"), "midpoint": quote.get("midpoint"),
        "last_trade": trade.get("price"),
    }


def get_option_chain(underlying, expiration_gte=None, expiration_lte=None,
                     strike_gte=None, strike_lte=None, limit=1_000):
    params = {"expiration_date.gte": expiration_gte, "expiration_date.lte": expiration_lte,
              "strike_price.gte": strike_gte, "strike_price.lte": strike_lte,
              "sort": "ticker", "order": "asc", "limit": min(limit, 250)}
    rows = paginate(f"/v3/snapshot/options/{underlying.upper()}", params,
                    max_pages=max(1, int(np.ceil(limit / 250))))
    frame = pd.DataFrame(_flatten_option(row) for row in rows)
    if frame.empty:
        return frame
    frame["expiration"] = pd.to_datetime(frame["expiration"])
    today = pd.Timestamp.now(tz="America/New_York").tz_localize(None).normalize()
    frame["dte"] = (frame["expiration"] - today).dt.days
    premium = frame["fmv"].combine_first(frame["midpoint"]).combine_first(frame["close"])
    spot, strike = frame["underlying_price"], frame["strike"]
    is_call = frame["contract_type"].eq("call")
    frame["distance_from_spot_pct"] = (strike / spot) - 1
    frame["intrinsic_value"] = np.where(is_call, np.maximum(spot - strike, 0), np.maximum(strike - spot, 0))
    frame["extrinsic_value"] = (premium - frame["intrinsic_value"]).clip(lower=0)
    frame["volume_oi_ratio"] = frame["volume"] / frame["open_interest"].replace(0, np.nan)
    frame["contract_notional"] = spot * frame["shares_per_contract"]
    frame["delta_adjusted_exposure"] = frame["delta"] * frame["contract_notional"]
    near_atm = frame["distance_from_spot_pct"].abs().le(0.01)
    frame["moneyness"] = np.select(
        [near_atm, is_call & (strike < spot), (~is_call) & (strike > spot)],
        ["ATM", "ITM", "ITM"], default="OTM"
    )
    return frame


def get_option_bars(option_ticker, start, end, multiplier=1, timespan="day", limit=50_000):
    path = f"/v2/aggs/ticker/{option_ticker}/range/{multiplier}/{timespan}/{start}/{end}"
    frame = results_frame(api_get(path, {"sort": "asc", "limit": limit})).rename(columns=BAR_COLUMNS)
    if "timestamp" in frame:
        frame["timestamp"] = unix_to_eastern(frame["timestamp"])
    return frame


def get_news(ticker, limit=20):
    rows = paginate("/v2/reference/news", {"ticker": ticker.upper(), "limit": limit,
                    "sort": "published_utc", "order": "desc"}, max_pages=1)
    return pd.json_normalize(rows)


def get_short_interest(ticker, limit=24):
    rows = paginate("/stocks/v1/short-interest", {"ticker": ticker.upper(), "limit": limit,
                    "sort": "settlement_date.desc"}, max_pages=1)
    return pd.DataFrame(rows).sort_values("settlement_date")

## Stock details and price data

Use the narrowest endpoint that answers the question:

- **Ticker details:** identity and company metadata
- **Aggregates:** reusable OHLCV history at custom timescales
- **Open/close:** a convenient single-session summary
- **Snapshot:** current session, previous session, FMV, and entitlement-dependent fields

In [ ]:
AAPL = "AAPL"
SPY = "SPY"
today = date.today()
one_year_ago = today - timedelta(days=365)

ticker_details = results_frame(api_get(f"/v3/reference/tickers/{AAPL}"))
detail_columns = ["ticker", "name", "primary_exchange", "sic_description", "market_cap",
                  "cik", "composite_figi", "list_date"]
ticker_details[[c for c in detail_columns if c in ticker_details]].T

In [ ]:
aapl_daily = get_stock_bars(AAPL, one_year_ago, today)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.72, 0.28], vertical_spacing=0.04)
fig.add_trace(
    go.Candlestick(
        x=aapl_daily.timestamp,
        open=aapl_daily.open,
        high=aapl_daily.high,
        low=aapl_daily.low,
        close=aapl_daily.close,
        name="AAPL",
    ),
    row=1,
    col=1,
)
fig.add_trace(go.Bar(x=aapl_daily.timestamp, y=aapl_daily.volume, name="Volume"), row=2, col=1)
fig.update_layout(title="AAPL - Daily OHLCV", height=560, xaxis_rangeslider_visible=False, showlegend=False)
fig.show()

In [ ]:
spy_daily = get_stock_bars(SPY, one_year_ago, today)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.72, 0.28], vertical_spacing=0.04)
fig.add_trace(
    go.Candlestick(
        x=spy_daily.timestamp,
        open=spy_daily.open,
        high=spy_daily.high,
        low=spy_daily.low,
        close=spy_daily.close,
        name="SPY",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(x=spy_daily.timestamp, y=spy_daily.volume, name="Volume"), row=2, col=1
)
fig.update_layout(title="SPY - Daily OHLCV", height=560, xaxis_rangeslider_visible=False, showlegend=False)
fig.show()

In [ ]:
# Different timescales use the same aggregate-bars endpoint.
latest_session = aapl_daily["timestamp"].max().date()
aapl_intraday = get_stock_bars(AAPL, latest_session, latest_session, multiplier=5, timespan="minute")
if not aapl_intraday.empty:
    px.line(aapl_intraday, x="timestamp", y="close", title=f"AAPL - 5-minute bars - {latest_session}").show()
else:
    print("No five-minute bars returned for the latest daily session.")

In [ ]:
# get open close data for latest session
open_close = api_get(f"/v1/open-close/{AAPL}/{latest_session}", {"adjusted": "true"})

# get previous session's summary
previous_day = results_frame(api_get(f"/v2/aggs/ticker/{AAPL}/prev", {"adjusted": "true"}))

# get isolated snapshot
single_snapshot = get_stock_snapshot(AAPL)

"""
get snapshot(s) for a range of tickers
note: requests receives {"tickers": "AAPL,MSFT,SPY"} and encodes this into a URL similar to 
/v2/snapshot/locale/us/markets/stocks/tickers?tickers=AAPL%2CMSFT%2CSPY. Massive receives this 
URL and decodes it into multiple data entries, returning something like this: 
{
    "status": "OK",
    "count": 3,
    "tickers": [
        {"ticker": "AAPL", ...},
        {"ticker": "MSFT", ...},
        {"ticker": "SPY", ...},
    ],
}
"""
watchlist_snapshot = results_frame(
    api_get("/v2/snapshot/locale/us/markets/stocks/tickers", {"tickers": "AAPL,MSFT,SPY"}),
    record_path="tickers",
)

open_close_summary = pd.DataFrame(
    [
        {
            "ticker": open_close.get("symbol"),
            "date": open_close.get("from"),
            "open": open_close.get("open"),
            "high": open_close.get("high"),
            "low": open_close.get("low"),
            "close": open_close.get("close"),
            "volume": open_close.get("volume"),
            "premarket": open_close.get("preMarket"),
            "after_hours": open_close.get("afterHours"),
        }
    ]
)

previous_day_summary = previous_day.rename(
    columns={
        "T": "ticker",
        "o": "open",
        "h": "high",
        "l": "low",
        "c": "close",
        "v": "volume",
        "vw": "vwap",
    }
)

watchlist_summary = watchlist_snapshot[
    [
        "ticker",
        "day.o",
        "day.h",
        "day.l",
        "day.c",
        "day.v",
        "todaysChange",
        "todaysChangePerc",
        "fmv",
    ]
]

display(open_close_summary)
display(previous_day_summary)
display(watchlist_summary)

In [ ]:
comparison = (
    aapl_daily[["timestamp", "close"]].rename(columns={"close": "AAPL"})
    .merge(spy_daily[["timestamp", "close"]].rename(columns={"close": "SPY"}), on="timestamp")
    .set_index("timestamp")
)
normalized = comparison.div(comparison.iloc[0]).mul(100).reset_index().melt("timestamp", var_name="ticker", value_name="indexed_value")
px.line(normalized, x="timestamp", y="indexed_value", color="ticker",
        title="Normalized performance - start = 100").show()

## Technical-indicator visualization

This section retrieves 250 days of AAPL price and indicator data, then presents them in three synchronized charts:

1. **Price trend:** closing price with the 20-day SMA and EMA.
2. **Momentum:** MACD and its signal line.
3. **Relative strength:** 14-day RSI with reference levels at 30 and 70.

In [ ]:
indicator_start = today - timedelta(days=250)
sma20 = get_indicator("sma", AAPL, window=20, start=indicator_start, end=today)
ema20 = get_indicator("ema", AAPL, window=20, start=indicator_start, end=today)
macd = get_indicator("macd", AAPL, start=indicator_start, end=today)
rsi14 = get_indicator("rsi", AAPL, window=14, start=indicator_start, end=today)

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, row_heights=[0.5, 0.25, 0.25], vertical_spacing=0.05)
recent_price = aapl_daily[aapl_daily.timestamp.dt.date >= indicator_start]
fig.add_trace(go.Scatter(x=recent_price.timestamp, y=recent_price.close, name="Close"), row=1, col=1)
if not sma20.empty: fig.add_trace(go.Scatter(x=sma20.timestamp, y=sma20.value, name="SMA 20"), row=1, col=1)
if not ema20.empty: fig.add_trace(go.Scatter(x=ema20.timestamp, y=ema20.value, name="EMA 20"), row=1, col=1)
if not macd.empty:
    fig.add_trace(go.Scatter(x=macd.timestamp, y=macd.value, name="MACD"), row=2, col=1)
    fig.add_trace(go.Scatter(x=macd.timestamp, y=macd.signal, name="Signal"), row=2, col=1)
if not rsi14.empty: fig.add_trace(go.Scatter(x=rsi14.timestamp, y=rsi14.value, name="RSI 14"), row=3, col=1)
fig.add_hline(y=70, line_dash="dot", row=3, col=1)
fig.add_hline(y=30, line_dash="dot", row=3, col=1)
fig.update_layout(title="AAPL - Polygon technical indicators", height=700)
fig.show()

## Financial statements and ratio analysis

This section retrieves eight quarters of AAPL income statements, balance sheets, and cash-flow statements.

It then:

- Displays important reported earnings fields
- Charts quarterly revenue and diluted EPS
- Shows recent balance-sheet and cash-flow metrics
- Organizes current ratios into valuation, profitability, liquidity, leverage, and cash-flow categories

These are reported financial results. The ratios endpoint provides the latest snapshot rather than a historical ratio series.

In [ ]:
income_q = get_financial_statements(AAPL, "income", "quarterly", limit=8)
balance_q = get_financial_statements(AAPL, "balance", "quarterly", limit=8)
cash_flow_q = get_financial_statements(AAPL, "cash_flow", "quarterly", limit=8)

earnings_columns = ["period_end", "fiscal_year", "fiscal_quarter", "revenue",
                    "diluted_earnings_per_share", "ebitda", "operating_income",
                    "net_income_loss_attributable_common_shareholders"]
earnings_trend = income_q[[c for c in earnings_columns if c in income_q]].copy()
earnings_trend.tail(8)

In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=income_q.period_end, y=income_q.revenue, name="Revenue"), secondary_y=False)
fig.add_trace(go.Scatter(x=income_q.period_end, y=income_q.diluted_earnings_per_share,
                         name="Diluted EPS", mode="lines+markers"), secondary_y=True)
fig.update_yaxes(title_text="Revenue ($)", secondary_y=False)
fig.update_yaxes(title_text="EPS ($/share)", secondary_y=True)
fig.update_layout(title="AAPL - Quarterly reported revenue and EPS", height=450)
fig.show()

balance_view = ["period_end", "cash_and_equivalents", "total_assets", "total_current_assets",
                "total_current_liabilities", "long_term_debt_and_capital_lease_obligations", "total_equity"]
cash_view = ["period_end", "net_cash_from_operating_activities",
             "purchase_of_property_plant_and_equipment", "net_cash_from_investing_activities",
             "net_cash_from_financing_activities"]
display(balance_q[[c for c in balance_view if c in balance_q]].tail(4))
display(cash_flow_q[[c for c in cash_view if c in cash_flow_q]].tail(4))

In [ ]:
ratios = get_ratios(AAPL)
ratio_groups = {
    "Valuation": ["price_to_earnings", "price_to_book", "price_to_sales", "ev_to_ebitda", "ev_to_sales"],
    "Profitability": ["return_on_assets", "return_on_equity", "earnings_per_share"],
    "Liquidity": ["current", "quick", "cash"],
    "Leverage": ["debt_to_equity", "enterprise_value"],
    "Cash flow": ["free_cash_flow", "price_to_cash_flow", "price_to_free_cash_flow"],
}
ratio_scorecard = pd.concat(
    [ratios[[c for c in fields if c in ratios]].T.rename(columns={0: group}) for group, fields in ratio_groups.items()],
    axis=1,
)
ratio_scorecard

### SPY options: contract discovery, pricing, IV, Greeks, and liquidity

This section demonstrates how to retrieve, organize, and analyze SPY options data.
- Find active contracts within selected expiration and strike ranges.
- Interpret OCC option symbols such as O:SPY261002C00761000.
- Dynamically select a relevant, liquid contract instead of hard-coding one.
- Retrieve daily prices, FMV, IV, open interest, break-even price, and Greeks.
- Calculate DTE, moneyness, intrinsic value, extrinsic value, and exposure metrics.
- Analyze the IV smile and compare Greeks across strikes.
- Apply volume and open-interest liquidity filters.
- Compare the selected option’s historical premium with SPY’s price.

In [ ]:
spy_snapshot = get_stock_snapshot(SPY)
spot = spy_snapshot.get("fmv") or spy_snapshot.get("day", {}).get("c") or spy_snapshot.get("prevDay", {}).get("c")

# get options contracts on SPY 20-45 DTE, limit 1000
start_expiry = today + timedelta(days=20)
end_expiry = today + timedelta(days=45)
contracts = list_option_contracts(
    SPY, start_expiry, end_expiry, strike_gte=round(spot * 0.90, 2),
    strike_lte=round(spot * 1.10, 2), limit=1_000,
)

print(f"SPY reference price: {spot:,.2f} - candidate contracts: {len(contracts):,}")
contracts.head()

In [ ]:
# get detailed options chain data on currently open options expiring in [start_expiry, end_expiry] w/ strikes 90-110% of current SPY spot price, limit 1000
chain = get_option_chain(
    SPY, start_expiry, end_expiry, strike_gte=round(spot * 0.90, 2),
    strike_lte=round(spot * 1.10, 2), limit=1_000,
)

chain_columns = ["ticker", "contract_type", "expiration", "dte", "strike", "underlying_price",
                 "fmv", "break_even", "volume", "open_interest", "iv", "delta", "gamma",
                 "theta", "vega", "moneyness", "distance_from_spot_pct"]
chain[[c for c in chain_columns if c in chain]].head(12)

In [ ]:
# heuristic filters for "reasonably" liquid options. don't copy this- it's just for the demo
liquid_chain = chain[
    chain["open_interest"].fillna(0).ge(100)
    & chain["volume"].fillna(0).ge(1)
    & chain["iv"].notna()
].copy()

# vol smile
plot_expiration = liquid_chain.groupby("expiration").size().idxmax()
smile = liquid_chain[liquid_chain.expiration.eq(plot_expiration)]
px.line(smile, x="strike", y="iv", color="contract_type", markers=True,
        title=f"SPY implied-volatility smile - {plot_expiration.date()}").show()

In [ ]:
greek_long = smile.melt(id_vars=["strike", "contract_type"],
                        value_vars=["delta", "gamma", "theta", "vega"],
                        var_name="greek", value_name="value").dropna()
px.line(greek_long, x="strike", y="value", color="contract_type", facet_row="greek",
        markers=True, title="SPY Greeks by strike", height=800).show()

In [ ]:
# Pick the most liquid contract within 2% of spot;
# otherwise use the nearest available strike.

near_money = chain[chain["distance_from_spot_pct"].abs().le(0.02)].copy()

selection_pool = near_money if not near_money.empty else chain.copy()

selection_pool["selection_oi"] = selection_pool["open_interest"].fillna(-1)

selection_pool["abs_distance_from_spot"] = selection_pool[
    "distance_from_spot_pct"
].abs()

selected = selection_pool.sort_values(
    ["selection_oi", "abs_distance_from_spot"],
    ascending=[False, True],
).iloc[0]

selected_ticker = selected["ticker"]

history_start = today - timedelta(days=45)

spy_history = get_stock_bars(
    SPY,
    history_start,
    today,
)

option_history = get_option_bars(
    selected_ticker,
    history_start,
    today,
)

print(f"Representative contract: {selected_ticker}")

if not option_history.empty and not spy_history.empty:
    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        row_heights=[0.35, 0.45, 0.20],
        vertical_spacing=0.06,
        subplot_titles=[
            "SPY Underlying Price",
            f"Option Premium - {selected_ticker}",
            "Option Volume",
        ],
    )

    fig.add_trace(
        go.Candlestick(
            x=spy_history["timestamp"],
            open=spy_history["open"],
            high=spy_history["high"],
            low=spy_history["low"],
            close=spy_history["close"],
            name="SPY",
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Candlestick(
            x=option_history["timestamp"],
            open=option_history["open"],
            high=option_history["high"],
            low=option_history["low"],
            close=option_history["close"],
            name="Option",
        ),
        row=2,
        col=1,
    )

    # Row 3: selected option volume
    fig.add_trace(
        go.Bar(
            x=option_history["timestamp"],
            y=option_history["volume"],
            name="Option Volume",
        ),
        row=3,
        col=1,
    )

    fig.update_yaxes(title_text="SPY Price ($)", row=1, col=1)
    fig.update_yaxes(title_text="Premium ($)", row=2, col=1)
    fig.update_yaxes(title_text="Volume", row=3, col=1)

    fig.update_xaxes(rangeslider_visible=False)

    fig.update_layout(
        title=f"SPY and Historical Option Premium - {selected_ticker}",
        height=850,
        showlegend=False,
    )

    fig.show()

## News, sentiment, positioning, and market context

Massive aggregates headlines, news sentiment, and related stocks on a per-ticker basis. News sentiment is article-level provider analysis.

Additionally, they provide scheduled releases of short interest on all stocks.  Short interest is reported on a scheduled cadence, not a real-time measure.

"Related companies" isn't necessarily a peer analysis grouping, but companies that trade similarly when ru.n through massive's models or share many news headlines.

In [ ]:
# news headlines and article links
news = get_news(AAPL, limit=25)
news_columns = ["published_utc", "title", "publisher.name", "article_url"]
display(news[[c for c in news_columns if c in news]].head(8))

# sentiment scores/"insights"
sentiment_rows = []
for article in paginate("/v2/reference/news", {"ticker": AAPL, "limit": 25,
                        "sort": "published_utc", "order": "desc"}, max_pages=1):
    for insight in article.get("insights", []) or []:
        if insight.get("ticker") == AAPL and insight.get("sentiment"):
            sentiment_rows.append({"title": article.get("title"), **insight})
sentiment = pd.DataFrame(sentiment_rows)
if not sentiment.empty:
    display(sentiment["sentiment"].value_counts().rename_axis("sentiment").to_frame("articles"))

In [ ]:
# time series short interest
short_interest = get_short_interest(AAPL)
if not short_interest.empty:
    short_interest["settlement_date"] = pd.to_datetime(short_interest["settlement_date"])
    px.line(short_interest, x="settlement_date", y="short_interest", markers=True,
            title="AAPL reported short interest").show()

# get related companies 
related = results_frame(api_get(f"/v1/related-companies/{AAPL}"))

# get live market status
market_status = api_get("/v1/marketstatus/now")

# get upcoming market holidays 
holidays = results_frame(api_get("/v1/marketstatus/upcoming"))

print("Related tickers:", ", ".join(related.get("ticker", pd.Series(dtype=str)).head(10)))
print("Current market state:", market_status.get("market"))
display(holidays.head(6))

## What to reuse

| Research need | Start here |
|---|---|
| Historical prices and returns | `get_stock_bars(...)` |
| Current stock monitoring | `get_stock_snapshot(...)` |
| Technical overlays | `get_indicator(...)` |
| Reported earnings and fundamentals | `get_financial_statements(...)` |
| Valuation/quality screen | `get_ratios(...)` |
| Contract universe | `list_option_contracts(...)` |
| Current IV/Greeks/OI research | `get_option_chain(...)` |
| Historical option premiums | `get_option_bars(...)` |
| News and sentiment | `get_news(...)` |
| Short interest | `get_short_interest(...)` |

### Before trusting a result

- Check whether prices are adjusted and which sessions are included.
- Convert timestamps explicitly to Eastern Time.
- Do not treat last option trades or FMV as executable prices.
- Expect missing Greeks, IV, volume, and open interest.
- Open interest reflects the prior session.
- Use flat files for large backtests.
- Never commit or share the club API key or licensed cached data.

## Troubleshooting

| Symptom | Likely cause | Response |
|---|---|---|
| `401` | Missing or invalid key | Reload `.env` |
| `403` | Endpoint or live-feed entitlement missing | Use an entitled endpoint |
| `404 Data not found` | No observations for that symbol/date | Try a liquid symbol or a valid trading session |
| Empty option bars | Contract did not trade in the interval | Choose a nearer-to-money, higher-volume contract |
| Missing Greeks or IV | Model could not produce a value | Drop missing rows or retain them explicitly; never fill |
| `[CACHE]` shown | A cached response was used | Use `cache_mode="live"` to force a refresh |
